# 05 — Model Evaluation & Deep Analysis

Standard metrics tell you how a model performs.
This notebook tells you *who it fails, how often, and why.*

All evaluation is performed on XGBoost as the selected deployment 
model, with Logistic Regression as the interpretable baseline 
for comparison throughout.

## 1. Confusion Matrices & Cost-Weighted Error Analysis

A confusion matrix tells you what kind of errors a model makes.
A cost-weighted confusion matrix tells you what those errors 
actually cost in the real world.

False negatives and false positives are not equal — in income 
prediction, wrongly denying a high earner (false negative) and 
wrongly approving a low earner (false positive) have different 
consequences depending on the downstream decision being made.

## 2. Bootstrap Confidence Intervals

Are XGBoost and LightGBM actually different models, or are we 
looking at noise in the third decimal place?

Bootstrap resampling (1000 iterations) gives us the distribution 
of ROC-AUC for each model. If the distributions overlap 
significantly, the models are statistically indistinguishable 
for practical purposes — and we should prefer the simpler one.

## 3. Subgroup Performance Breakdown

Overall metrics hide subgroup failures. A model with 84% accuracy 
can simultaneously be excellent for one group and poor for another.

For each protected attribute — sex, race, age_group — we compute 
full classification metrics independently:
- Precision, Recall, F1, ROC-AUC per group
- Visualized as a heatmap: groups × metrics

The question: does the model fail differently for different groups, 
and is that failure pattern random or systematic?

## 4. False Positive & False Negative Profiles

Misclassifications are not random. This section builds demographic 
profiles of the people the model gets wrong:

- **False Positives** — predicted >50K, actually <=50K
- **False Negatives** — predicted <=50K, actually >50K

Are false negatives concentrated in specific demographic groups?
What features characterize the hardest cases — predictions 
close to the 0.5 decision boundary?

This is the question a policy maker would ask. We answer it with data.

## 5. Threshold Analysis — Fairness Without Retraining

The 0.5 decision threshold is arbitrary. Moving it changes the 
precision-recall tradeoff — and also changes fairness metrics.

We plot how precision, recall, F1, and Demographic Parity Difference 
change as the threshold moves from 0.1 to 0.9, and identify:
- Threshold that maximizes F1
- Threshold that minimizes DPD
- Threshold that best balances performance and fairness

Key question: can we improve fairness through threshold adjustment 
alone, without retraining? If yes, that is a practical finding 
with immediate deployment implications.

## 6. Model Disagreement Analysis

Where do XGBoost and Logistic Regression disagree?

Cases where a complex model and a simple model give different 
predictions are the most revealing — they show where non-linear 
patterns matter and where the data has genuine ambiguity.

Profiling disagreement cases demographically tells us whether 
the added complexity of XGBoost benefits all groups equally 
or primarily serves the majority group.

## 7. Deployment Verdict

One page. Plain language.

If this model were deployed tomorrow:
- Which model, and why
- Who is most at risk from its errors
- What monitoring would be required
- What would change with more time or data

This is not a technical summary — it is a responsible deployment 
assessment written for a decision maker, not a data scientist.

## Setup & Imports

In [11]:
import pandas as pd
import numpy as np
import yaml
import warnings
warnings.filterwarnings("ignore")

import matplotlib.pyplot as plt
import seaborn as sns

import joblib

from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, brier_score_loss,
    log_loss,
    roc_curve, precision_recall_curve, confusion_matrix,
    classification_report,
)
from sklearn.calibration import calibration_curve
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.utils import resample

from scipy import stats
from scipy.stats import wilcoxon

import xgboost as xgb
import lightgbm as lgb
from fairlearn.reductions import ExponentiatedGradient, DemographicParity
from fairlearn.metrics import demographic_parity_difference

In [12]:
with open("../config.yaml", "r") as f:
    config = yaml.safe_load(f)

sns.set_theme(style="whitegrid")
sns.set_palette("husl")
np.random.seed(42)

print("Libraries loaded.")

Libraries loaded.


In [13]:
# Extract protected attributes
PROTECTED_ATTRS = config['protected_attributes']
TARGET = config['target']['column']
MODELS_PATH = config['paths']['models']
FIGURES_PATH = config['paths']['figures']

print(f"Configuration loaded.")
print(f"Protected attributes: {PROTECTED_ATTRS}")
print(f"Target variable: {TARGET}")

Configuration loaded.
Protected attributes: ['sex', 'race', 'age', 'marital-status', 'native-country']
Target variable: income


## 1. Load Data & Models

In [14]:
# Load processed data
train = pd.read_csv("../data/processed/train.csv")
test = pd.read_csv("../data/processed/test.csv")

# Separate features and target
X_train = train.drop(columns=[TARGET])
y_train = train[TARGET]
X_test = test.drop(columns=[TARGET])
y_test = test[TARGET]

print(f"Training set: {X_train.shape}")
print(f"Test set: {X_test.shape}")
print(f"\nTarget distribution (train): {y_train.value_counts(normalize=True)}")
print(f"\nTarget distribution (test): {y_test.value_counts(normalize=True)}")

Training set: (32561, 18)
Test set: (16281, 18)

Target distribution (train): income
0    0.75919
1    0.24081
Name: proportion, dtype: float64

Target distribution (test): income
0    0.763774
1    0.236226
Name: proportion, dtype: float64


In [15]:
# Load trained models
# Expected models: Logistic Regression, Random Forest, XGBoost, LightGBM, Neural Network, Calibrated variants

models = {}
model_names = [
    'logistic_regression',
    'random_forest',
    'xgboost',
    'lightgbm',
    'neural_network',
    'calibrated_model'
]

# Try to load each model
for model_name in model_names:
    try:
        model_path = f"../{MODELS_PATH}{model_name}.pkl"
        models[model_name] = joblib.load(model_path)
        print(f"✓ Loaded: {model_name}")
    except FileNotFoundError:
        print(f"✗ Not found: {model_name}")

print(f"\nTotal models loaded: {len(models)}")

NameError: name 'joblib' is not defined

## 2. Generate Predictions & Probabilities

In [ ]:
# Generate predictions for all models
predictions = {}
probabilities = {}

for name, model in models.items():
    try:
        # Get class predictions
        predictions[name] = model.predict(X_test)
        
        # Get probability predictions (for positive class)
        if hasattr(model, 'predict_proba'):
            probabilities[name] = model.predict_proba(X_test)[:, 1]
        elif hasattr(model, 'decision_function'):
            # For models without predict_proba, use decision_function
            from scipy.special import expit
            probabilities[name] = expit(model.decision_function(X_test))
        
        print(f"✓ Generated predictions for: {name}")
    except Exception as e:
        print(f"✗ Error with {name}: {str(e)}")

print(f"\nPredictions generated for {len(predictions)} models.")

## 3. Basic Performance Metrics

In [ ]:
# Calculate comprehensive metrics for each model
metrics_df = []

for name in predictions.keys():
    y_pred = predictions[name]
    y_prob = probabilities.get(name, None)
    
    metric_row = {
        'Model': name,
        'Accuracy': accuracy_score(y_test, y_pred),
        'Precision': precision_score(y_test, y_pred, zero_division=0),
        'Recall': recall_score(y_test, y_pred, zero_division=0),
        'F1': f1_score(y_test, y_pred, zero_division=0),
    }
    
    if y_prob is not None:
        metric_row['ROC-AUC'] = roc_auc_score(y_test, y_prob)
        metric_row['PR-AUC'] = average_precision_score(y_test, y_prob)
        metric_row['Brier Score'] = brier_score_loss(y_test, y_prob)
        metric_row['Log Loss'] = log_loss(y_test, y_prob)
    
    metrics_df.append(metric_row)

metrics_df = pd.DataFrame(metrics_df)
metrics_df = metrics_df.round(4)

print("\n=== Model Performance Summary ===")
display(metrics_df)

# Save metrics
metrics_df.to_csv("../reports/model_metrics.csv", index=False)
print("\n✓ Metrics saved to reports/model_metrics.csv")

In [ ]:
# Visualize performance metrics
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

# Plot key metrics
metrics_to_plot = ['Accuracy', 'F1', 'ROC-AUC', 'Brier Score']

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx // 2, idx % 2]
    if metric in metrics_df.columns:
        data = metrics_df.sort_values(metric, ascending=(metric == 'Brier Score'))
        ax.barh(data['Model'], data[metric])
        ax.set_xlabel(metric, fontsize=12)
        ax.set_title(f'Model Comparison: {metric}', fontsize=14, fontweight='bold')
        ax.grid(axis='x', alpha=0.3)

plt.tight_layout()
plt.savefig(f"../{FIGURES_PATH}model_performance_comparison.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ Performance comparison chart saved.")

## 4. Calibration Analysis

Well-calibrated models produce probability estimates that match observed frequencies.
If a model predicts 70% probability for 100 instances, approximately 70 should be positive.

**Calibration Metrics:**
- **Calibration Curve**: Visual assessment of calibration quality
- **Expected Calibration Error (ECE)**: Quantitative measure of calibration
- **Brier Score**: Measures both calibration and refinement

In [ ]:
def expected_calibration_error(y_true, y_prob, n_bins=10):
    """
    Calculate Expected Calibration Error (ECE).
    
    ECE is the weighted average of the absolute difference between
    predicted probabilities and observed frequencies across bins.
    """
    prob_true, prob_pred = calibration_curve(y_true, y_prob, n_bins=n_bins, strategy='uniform')
    
    # Bin predictions
    bins = np.linspace(0, 1, n_bins + 1)
    bin_indices = np.digitize(y_prob, bins[1:-1])
    
    # Calculate ECE
    ece = 0.0
    for i in range(n_bins):
        mask = bin_indices == i
        if np.sum(mask) > 0:
            bin_accuracy = np.mean(y_true[mask])
            bin_confidence = np.mean(y_prob[mask])
            bin_size = np.sum(mask) / len(y_true)
            ece += bin_size * np.abs(bin_accuracy - bin_confidence)
    
    return ece

print("✓ Calibration error function defined.")

In [ ]:
# Calculate ECE for all models
calibration_scores = []

for name, y_prob in probabilities.items():
    ece = expected_calibration_error(y_test.values, y_prob, n_bins=10)
    brier = brier_score_loss(y_test, y_prob)
    
    calibration_scores.append({
        'Model': name,
        'ECE': ece,
        'Brier Score': brier
    })

calibration_df = pd.DataFrame(calibration_scores).round(4)
calibration_df = calibration_df.sort_values('ECE')

print("\n=== Model Calibration Summary ===")
print("Lower ECE and Brier Score indicate better calibration.\n")
display(calibration_df)

In [ ]:
# Plot calibration curves for all models
n_models = len(probabilities)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten() if n_models > 1 else [axes]

for idx, (name, y_prob) in enumerate(probabilities.items()):
    ax = axes[idx]
    
    # Calculate calibration curve
    prob_true, prob_pred = calibration_curve(y_test, y_prob, n_bins=10, strategy='uniform')
    
    # Plot calibration curve
    ax.plot(prob_pred, prob_true, marker='o', linewidth=2, label=name)
    ax.plot([0, 1], [0, 1], 'k--', label='Perfect Calibration')
    
    # Add ECE to plot
    ece = calibration_df[calibration_df['Model'] == name]['ECE'].values[0]
    ax.text(0.05, 0.95, f'ECE: {ece:.4f}', transform=ax.transAxes,
            verticalalignment='top', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    
    ax.set_xlabel('Mean Predicted Probability', fontsize=11)
    ax.set_ylabel('Fraction of Positives', fontsize=11)
    ax.set_title(f'Calibration Curve: {name}', fontsize=12, fontweight='bold')
    ax.legend(loc='lower right')
    ax.grid(alpha=0.3)

# Remove extra subplots
for idx in range(n_models, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig(f"../{FIGURES_PATH}calibration_curves.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ Calibration curves saved.")

## 5. Confidence Intervals via Bootstrap

Point estimates can be misleading. We use bootstrapping to estimate confidence intervals
for our performance metrics, providing a measure of uncertainty.

In [ ]:
def bootstrap_metric(y_true, y_pred, y_prob=None, metric_name='accuracy', n_bootstrap=1000, confidence=0.95):
    """
    Calculate bootstrap confidence intervals for a metric.
    
    Parameters:
    -----------
    y_true : array-like
        True labels
    y_pred : array-like
        Predicted labels
    y_prob : array-like, optional
        Predicted probabilities (required for some metrics)
    metric_name : str
        Name of metric to calculate
    n_bootstrap : int
        Number of bootstrap iterations
    confidence : float
        Confidence level (e.g., 0.95 for 95% CI)
    
    Returns:
    --------
    dict with 'mean', 'ci_lower', 'ci_upper'
    """
    n_samples = len(y_true)
    scores = []
    
    for _ in range(n_bootstrap):
        # Bootstrap sample
        indices = np.random.choice(n_samples, n_samples, replace=True)
        y_true_boot = y_true[indices]
        y_pred_boot = y_pred[indices]
        
        # Calculate metric
        if metric_name == 'accuracy':
            score = accuracy_score(y_true_boot, y_pred_boot)
        elif metric_name == 'f1':
            score = f1_score(y_true_boot, y_pred_boot, zero_division=0)
        elif metric_name == 'roc_auc' and y_prob is not None:
            y_prob_boot = y_prob[indices]
            try:
                score = roc_auc_score(y_true_boot, y_prob_boot)
            except:
                continue
        else:
            continue
        
        scores.append(score)
    
    # Calculate confidence interval
    alpha = 1 - confidence
    ci_lower = np.percentile(scores, 100 * alpha / 2)
    ci_upper = np.percentile(scores, 100 * (1 - alpha / 2))
    
    return {
        'mean': np.mean(scores),
        'ci_lower': ci_lower,
        'ci_upper': ci_upper,
        'std': np.std(scores)
    }

print("✓ Bootstrap function defined.")

In [ ]:
# Calculate confidence intervals for key metrics
ci_results = []

print("Calculating bootstrap confidence intervals (this may take a moment)...\n")

for name in predictions.keys():
    y_pred = predictions[name]
    y_prob = probabilities.get(name, None)
    
    # Accuracy CI
    acc_ci = bootstrap_metric(y_test.values, y_pred, metric_name='accuracy', n_bootstrap=1000)
    
    # F1 CI
    f1_ci = bootstrap_metric(y_test.values, y_pred, metric_name='f1', n_bootstrap=1000)
    
    # ROC-AUC CI (if probabilities available)
    roc_auc_ci = None
    if y_prob is not None:
        roc_auc_ci = bootstrap_metric(y_test.values, y_pred, y_prob, metric_name='roc_auc', n_bootstrap=1000)
    
    ci_results.append({
        'Model': name,
        'Accuracy': f"{acc_ci['mean']:.4f}",
        'Accuracy CI': f"[{acc_ci['ci_lower']:.4f}, {acc_ci['ci_upper']:.4f}]",
        'F1': f"{f1_ci['mean']:.4f}",
        'F1 CI': f"[{f1_ci['ci_lower']:.4f}, {f1_ci['ci_upper']:.4f}]",
        'ROC-AUC': f"{roc_auc_ci['mean']:.4f}" if roc_auc_ci else 'N/A',
        'ROC-AUC CI': f"[{roc_auc_ci['ci_lower']:.4f}, {roc_auc_ci['ci_upper']:.4f}]" if roc_auc_ci else 'N/A'
    })
    
    print(f"✓ Completed: {name}")

ci_df = pd.DataFrame(ci_results)

print("\n=== Performance Metrics with 95% Confidence Intervals ===")
display(ci_df)

ci_df.to_csv("../reports/confidence_intervals.csv", index=False)
print("\n✓ Confidence intervals saved.")

## 6. Statistical Significance Testing

Are the performance differences between models statistically significant?
We use McNemar's test for comparing classifiers on the same test set.

In [ ]:
def mcnemar_test(y_true, y_pred1, y_pred2):
    """
    Perform McNemar's test to compare two classifiers.
    
    McNemar's test is appropriate for paired nominal data (same test set).
    Null hypothesis: The two classifiers have the same error rate.
    
    Returns p-value.
    """
    # Create contingency table
    correct1 = (y_pred1 == y_true)
    correct2 = (y_pred2 == y_true)
    
    n00 = np.sum(~correct1 & ~correct2)  # Both wrong
    n01 = np.sum(~correct1 & correct2)   # Model 1 wrong, Model 2 correct
    n10 = np.sum(correct1 & ~correct2)   # Model 1 correct, Model 2 wrong
    n11 = np.sum(correct1 & correct2)    # Both correct
    
    # McNemar's test statistic (with continuity correction)
    if n01 + n10 == 0:
        return 1.0  # No discordant pairs
    
    statistic = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)
    p_value = 1 - stats.chi2.cdf(statistic, 1)
    
    return p_value

print("✓ McNemar's test function defined.")

In [ ]:
# Perform pairwise significance tests
model_names_list = list(predictions.keys())
n_models = len(model_names_list)

# Create matrix for p-values
p_value_matrix = np.ones((n_models, n_models))

for i, model1 in enumerate(model_names_list):
    for j, model2 in enumerate(model_names_list):
        if i < j:  # Only compute upper triangle
            p_value = mcnemar_test(
                y_test.values,
                predictions[model1],
                predictions[model2]
            )
            p_value_matrix[i, j] = p_value
            p_value_matrix[j, i] = p_value  # Symmetric

# Create DataFrame
p_value_df = pd.DataFrame(p_value_matrix, 
                          index=model_names_list, 
                          columns=model_names_list)
p_value_df = p_value_df.round(4)

print("\n=== McNemar's Test P-Values (Pairwise Comparisons) ===")
print("P-value < 0.05 indicates statistically significant difference.\n")
display(p_value_df)

# Save results
p_value_df.to_csv("../reports/significance_tests.csv")
print("\n✓ Significance test results saved.")

In [ ]:
# Visualize significance test results
plt.figure(figsize=(10, 8))
sns.heatmap(p_value_df, annot=True, cmap='RdYlGn_r', 
            vmin=0, vmax=0.1, center=0.05,
            square=True, linewidths=1, cbar_kws={'label': 'P-value'})
plt.title('Statistical Significance of Model Differences\n(McNemar\'s Test)', 
          fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f"../{FIGURES_PATH}significance_heatmap.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ Significance heatmap saved.")

## 7. Error Analysis

Understanding *where* and *why* models fail is crucial for improvement.
We analyze misclassification patterns and identify challenging cases.

In [ ]:
# Analyze confusion matrices
n_models = len(predictions)
n_cols = 3
n_rows = (n_models + n_cols - 1) // n_cols

fig, axes = plt.subplots(n_rows, n_cols, figsize=(15, 5 * n_rows))
axes = axes.flatten() if n_models > 1 else [axes]

for idx, (name, y_pred) in enumerate(predictions.items()):
    ax = axes[idx]
    
    # Compute confusion matrix
    cm = confusion_matrix(y_test, y_pred)
    
    # Normalize to show percentages
    cm_normalized = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
    
    # Plot
    sns.heatmap(cm_normalized, annot=True, fmt='.2%', cmap='Blues', 
                ax=ax, cbar_kws={'label': 'Proportion'})
    ax.set_title(f'Confusion Matrix: {name}', fontsize=12, fontweight='bold')
    ax.set_ylabel('True Label')
    ax.set_xlabel('Predicted Label')
    
    # Add counts as text
    for i in range(2):
        for j in range(2):
            ax.text(j + 0.5, i + 0.7, f'({cm[i, j]})', 
                   ha='center', va='center', fontsize=9, color='gray')

# Remove extra subplots
for idx in range(n_models, len(axes)):
    fig.delaxes(axes[idx])

plt.tight_layout()
plt.savefig(f"../{FIGURES_PATH}confusion_matrices.png", dpi=300, bbox_inches='tight')
plt.show()

print("✓ Confusion matrices saved.")

In [ ]:
# Identify consistently misclassified instances
# These are the "hard cases" that all models struggle with

# Create a matrix of correct/incorrect predictions
error_matrix = pd.DataFrame()

for name, y_pred in predictions.items():
    error_matrix[name] = (y_pred != y_test.values).astype(int)

# Count how many models got each instance wrong
error_counts = error_matrix.sum(axis=1)

# Instances where all models failed
all_wrong = error_counts == n_models
print(f"\nInstances where ALL models failed: {all_wrong.sum()} ({100*all_wrong.sum()/len(y_test):.2f}%)")

# Instances where most models failed (>50%)
most_wrong = error_counts > (n_models / 2)
print(f"Instances where MOST models failed: {most_wrong.sum()} ({100*most_wrong.sum()/len(y_test):.2f}%)")

# Instances where all models succeeded
all_right = error_counts == 0
print(f"Instances where ALL models succeeded: {all_right.sum()} ({100*all_right.sum()/len(y_test):.2f}%)")

# Distribution of error counts
plt.figure(figsize=(10, 6))
error_counts.value_counts().sort_index().plot(kind='bar', color='steelblue')
plt.xlabel('Number of Models that Misclassified', fontsize=12)
plt.ylabel('Number of Instances', fontsize=12)
plt.title('Distribution of Misclassifications Across Models', fontsize=14, fontweight='bold')
plt.xticks(rotation=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f"../{FIGURES_PATH}error_distribution.png", dpi=300, bbox_inches='tight')
plt.show()

print("\n✓ Error analysis completed.")

In [ ]:
# Examine characteristics of hard cases
# Add error count to test data for analysis
test_with_errors = test.copy()
test_with_errors['error_count'] = error_counts.values
test_with_errors['all_wrong'] = all_wrong.values
test_with_errors['all_right'] = all_right.values

print("\n=== Characteristics of Hard Cases (All Models Wrong) ===")

if all_wrong.sum() > 0:
    hard_cases = test_with_errors[test_with_errors['all_wrong']]
    
    # Show distribution across categorical features
    categorical_features = X_test.select_dtypes(include=['object']).columns[:5]  # First 5 categorical
    
    fig, axes = plt.subplots(1, len(categorical_features), figsize=(20, 4))
    
    for idx, feature in enumerate(categorical_features):
        if feature in hard_cases.columns:
            ax = axes[idx] if len(categorical_features) > 1 else axes
            
            # Compare distribution in hard cases vs all cases
            hard_dist = hard_cases[feature].value_counts(normalize=True)
            all_dist = test_with_errors[feature].value_counts(normalize=True)
            
            comparison = pd.DataFrame({
                'Hard Cases': hard_dist,
                'All Cases': all_dist
            }).fillna(0)
            
            comparison.plot(kind='bar', ax=ax)
            ax.set_title(f'{feature}', fontsize=11)
            ax.set_xlabel('')
            ax.tick_params(axis='x', rotation=45)
            ax.legend(fontsize=9)
    
    plt.tight_layout()
    plt.savefig(f"../{FIGURES_PATH}hard_cases_analysis.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Hard cases analysis completed.")
else:
    print("No instances where all models failed.")

## 8. Subgroup Performance Analysis

Model performance may vary across demographic groups (protected attributes).
We analyze performance by sex, race, age, and other sensitive attributes to identify
potential fairness issues.

In [ ]:
# Prepare test data with predictions for subgroup analysis
subgroup_df = test.copy()

# Add predictions from each model
for name, y_pred in predictions.items():
    subgroup_df[f'{name}_pred'] = y_pred
    if name in probabilities:
        subgroup_df[f'{name}_prob'] = probabilities[name]

print("✓ Subgroup analysis data prepared.")
print(f"\nProtected attributes to analyze: {PROTECTED_ATTRS}")

In [ ]:
def analyze_subgroup_performance(df, protected_attr, model_name, target_col):
    """
    Analyze model performance across subgroups of a protected attribute.
    
    Returns DataFrame with performance metrics for each subgroup.
    """
    if protected_attr not in df.columns:
        print(f"Warning: {protected_attr} not found in data")
        return None
    
    pred_col = f'{model_name}_pred'
    prob_col = f'{model_name}_prob'
    
    if pred_col not in df.columns:
        print(f"Warning: {pred_col} not found in data")
        return None
    
    results = []
    
    for group in df[protected_attr].unique():
        mask = df[protected_attr] == group
        
        if mask.sum() < 10:  # Skip very small groups
            continue
        
        y_true = df.loc[mask, target_col]
        y_pred = df.loc[mask, pred_col]
        
        metrics = {
            'Group': group,
            'Size': mask.sum(),
            'Prevalence': y_true.mean(),
            'Accuracy': accuracy_score(y_true, y_pred),
            'Precision': precision_score(y_true, y_pred, zero_division=0),
            'Recall': recall_score(y_true, y_pred, zero_division=0),
            'F1': f1_score(y_true, y_pred, zero_division=0)
        }
        
        if prob_col in df.columns:
            y_prob = df.loc[mask, prob_col]
            try:
                metrics['ROC-AUC'] = roc_auc_score(y_true, y_prob)
            except:
                metrics['ROC-AUC'] = np.nan
        
        results.append(metrics)
    
    return pd.DataFrame(results)

print("✓ Subgroup analysis function defined.")

In [ ]:
# Analyze performance across protected attributes
# Focus on one representative model for detailed analysis

if len(models) > 0:
    # Select the first model for detailed subgroup analysis
    selected_model = list(models.keys())[0]
    
    print(f"\n=== Subgroup Analysis: {selected_model} ===")
    print("\nAnalyzing performance across protected attributes...\n")
    
    for attr in PROTECTED_ATTRS:
        if attr in subgroup_df.columns:
            print(f"\n--- {attr.upper()} ---")
            
            results = analyze_subgroup_performance(
                subgroup_df, attr, selected_model, TARGET
            )
            
            if results is not None:
                display(results.round(4))
                
                # Save results
                results.to_csv(f"../reports/subgroup_{attr}_{selected_model}.csv", index=False)
            else:
                print(f"Could not analyze {attr}")
        else:
            print(f"\n{attr} not found in data")
else:
    print("No models available for subgroup analysis.")

In [ ]:
# Visualize subgroup performance disparities
# Focus on key protected attributes: sex and race

key_attrs = ['sex', 'race']
available_attrs = [attr for attr in key_attrs if attr in subgroup_df.columns]

if len(models) > 0 and len(available_attrs) > 0:
    selected_model = list(models.keys())[0]
    
    fig, axes = plt.subplots(len(available_attrs), 1, figsize=(12, 6 * len(available_attrs)))
    if len(available_attrs) == 1:
        axes = [axes]
    
    for idx, attr in enumerate(available_attrs):
        ax = axes[idx]
        
        results = analyze_subgroup_performance(
            subgroup_df, attr, selected_model, TARGET
        )
        
        if results is not None:
            # Plot accuracy, precision, recall, F1
            metrics_to_plot = ['Accuracy', 'Precision', 'Recall', 'F1']
            x = np.arange(len(results))
            width = 0.2
            
            for i, metric in enumerate(metrics_to_plot):
                if metric in results.columns:
                    ax.bar(x + i*width, results[metric], width, label=metric)
            
            ax.set_xlabel('Subgroup', fontsize=12)
            ax.set_ylabel('Score', fontsize=12)
            ax.set_title(f'Performance by {attr.title()} - {selected_model}', 
                        fontsize=14, fontweight='bold')
            ax.set_xticks(x + width * 1.5)
            ax.set_xticklabels(results['Group'], rotation=45, ha='right')
            ax.legend()
            ax.grid(axis='y', alpha=0.3)
            ax.set_ylim([0, 1])
    
    plt.tight_layout()
    plt.savefig(f"../{FIGURES_PATH}subgroup_performance.png", dpi=300, bbox_inches='tight')
    plt.show()
    
    print("✓ Subgroup performance visualization saved.")
else:
    print("Protected attributes not available for visualization.")

## 9. Performance Disparity Metrics

Quantify performance disparities across groups using:
- **Max Difference**: Maximum performance gap between any two groups
- **Coefficient of Variation**: Relative variability in performance
- **Statistical Tests**: Whether differences are statistically significant

In [ ]:
def calculate_disparity_metrics(subgroup_results, metric_name='Accuracy'):
    """
    Calculate disparity metrics for a given performance metric across subgroups.
    """
    if metric_name not in subgroup_results.columns:
        return None
    
    values = subgroup_results[metric_name].values
    
    return {
        'Metric': metric_name,
        'Min': values.min(),
        'Max': values.max(),
        'Mean': values.mean(),
        'Std': values.std(),
        'Max Difference': values.max() - values.min(),
        'Coefficient of Variation': values.std() / values.mean() if values.mean() > 0 else np.nan
    }

# Calculate disparity metrics for each protected attribute
if len(models) > 0:
    selected_model = list(models.keys())[0]
    
    print(f"\n=== Performance Disparity Analysis: {selected_model} ===")
    
    for attr in PROTECTED_ATTRS:
        if attr in subgroup_df.columns:
            results = analyze_subgroup_performance(
                subgroup_df, attr, selected_model, TARGET
            )
            
            if results is not None and len(results) > 1:
                print(f"\n--- {attr.upper()} ---")
                
                disparities = []
                for metric in ['Accuracy', 'Precision', 'Recall', 'F1']:
                    if metric in results.columns:
                        disp = calculate_disparity_metrics(results, metric)
                        if disp:
                            disparities.append(disp)
                
                if disparities:
                    disp_df = pd.DataFrame(disparities).round(4)
                    display(disp_df)

print("\n✓ Disparity analysis completed.")

## 10. Model Comparison & Selection

Synthesize all evaluation dimensions to make an informed model selection decision.
Consider:
- Overall performance (accuracy, F1, ROC-AUC)
- Calibration quality (ECE, Brier score)
- Fairness (subgroup performance disparities)
- Statistical significance of differences
- Interpretability and deployment considerations

In [ ]:
# Create comprehensive model comparison table
comparison_data = []

for name in models.keys():
    row = {'Model': name}
    
    # Performance metrics
    if name in predictions:
        perf = metrics_df[metrics_df['Model'] == name]
        if len(perf) > 0:
            row.update(perf.iloc[0].to_dict())
    
    # Calibration metrics
    if name in probabilities:
        cal = calibration_df[calibration_df['Model'] == name]
        if len(cal) > 0:
            row['ECE'] = cal.iloc[0]['ECE']
    
    comparison_data.append(row)

comparison_df = pd.DataFrame(comparison_data)

print("\n=== Comprehensive Model Comparison ===")
display(comparison_df)

# Save comparison
comparison_df.to_csv("../reports/model_comparison_summary.csv", index=False)
print("\n✓ Model comparison saved.")

In [ ]:
# Create radar chart for multi-dimensional model comparison
if len(models) >= 2:
    from math import pi
    
    # Select metrics for radar chart (normalized to 0-1)
    metrics_for_radar = ['Accuracy', 'F1', 'ROC-AUC']
    
    # Add inverse ECE (so higher is better)
    if 'ECE' in comparison_df.columns:
        comparison_df['Calibration'] = 1 - comparison_df['ECE']
        metrics_for_radar.append('Calibration')
    
    # Filter available metrics
    available_metrics = [m for m in metrics_for_radar if m in comparison_df.columns]
    
    if len(available_metrics) >= 3:
        # Number of variables
        num_vars = len(available_metrics)
        angles = [n / float(num_vars) * 2 * pi for n in range(num_vars)]
        angles += angles[:1]
        
        # Plot
        fig, ax = plt.subplots(figsize=(10, 10), subplot_kw=dict(projection='polar'))
        
        # Plot each model
        for idx, row in comparison_df.iterrows():
            values = row[available_metrics].values.flatten().tolist()
            values += values[:1]
            ax.plot(angles, values, 'o-', linewidth=2, label=row['Model'])
            ax.fill(angles, values, alpha=0.15)
        
        # Set labels
        ax.set_xticks(angles[:-1])
        ax.set_xticklabels(available_metrics, size=11)
        ax.set_ylim(0, 1)
        ax.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
        ax.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'], size=9)
        ax.grid(True)
        
        plt.legend(loc='upper right', bbox_to_anchor=(1.3, 1.1))
        plt.title('Multi-Dimensional Model Comparison', 
                 size=16, fontweight='bold', pad=20)
        
        plt.tight_layout()
        plt.savefig(f"../{FIGURES_PATH}model_radar_chart.png", dpi=300, bbox_inches='tight')
        plt.show()
        
        print("✓ Radar chart saved.")
    else:
        print("Not enough metrics for radar chart.")
else:
    print("Need at least 2 models for comparison.")

## 11. Conclusions & Recommendations

Synthesize findings and provide recommendations for model selection and deployment.

In [ ]:
print("\n" + "="*80)
print("EVALUATION SUMMARY")
print("="*80)

if len(models) > 0:
    print(f"\nTotal models evaluated: {len(models)}")
    print(f"Models: {', '.join(models.keys())}")
    
    # Best model by different criteria
    print("\n--- Best Models by Criterion ---")
    
    if 'Accuracy' in comparison_df.columns:
        best_acc = comparison_df.loc[comparison_df['Accuracy'].idxmax()]
        print(f"Highest Accuracy: {best_acc['Model']} ({best_acc['Accuracy']:.4f})")
    
    if 'F1' in comparison_df.columns:
        best_f1 = comparison_df.loc[comparison_df['F1'].idxmax()]
        print(f"Highest F1 Score: {best_f1['Model']} ({best_f1['F1']:.4f})")
    
    if 'ROC-AUC' in comparison_df.columns:
        best_auc = comparison_df.loc[comparison_df['ROC-AUC'].idxmax()]
        print(f"Highest ROC-AUC: {best_auc['Model']} ({best_auc['ROC-AUC']:.4f})")
    
    if 'ECE' in comparison_df.columns:
        best_cal = comparison_df.loc[comparison_df['ECE'].idxmin()]
        print(f"Best Calibration (Lowest ECE): {best_cal['Model']} ({best_cal['ECE']:.4f})")
    
    print("\n--- Key Findings ---")
    print("• Performance metrics calculated with 95% confidence intervals")
    print("• Statistical significance tested using McNemar's test")
    print("• Calibration quality assessed using ECE and Brier score")
    print("• Subgroup performance analyzed across protected attributes")
    print("• Error patterns identified for model improvement")
    
    print("\n--- Recommendations ---")
    print("1. Review calibration curves - well-calibrated probabilities are crucial for decision-making")
    print("2. Examine subgroup disparities - ensure fair performance across demographic groups")
    print("3. Consider statistical significance - small performance differences may not be meaningful")
    print("4. Analyze hard cases - understand where all models struggle")
    print("5. Balance performance and fairness - highest accuracy != best model")
    
    print("\n--- Next Steps ---")
    print("• Conduct fairness audit (if not already done)")
    print("• Implement fairness mitigation if disparities are significant")
    print("• Consider model calibration techniques if ECE is high")
    print("• Document model limitations and appropriate use cases")
    print("• Establish monitoring plan for production deployment")
else:
    print("\nNo models loaded for evaluation.")
    print("Please ensure models are trained and saved before running this notebook.")

print("\n" + "="*80)
print("✓ EVALUATION COMPLETE")
print("="*80)

---

## Report Generated

**Outputs:**
- `reports/model_metrics.csv` - Comprehensive performance metrics
- `reports/confidence_intervals.csv` - Bootstrap confidence intervals
- `reports/significance_tests.csv` - Statistical significance tests
- `reports/model_comparison_summary.csv` - Overall comparison
- `reports/figures/` - All visualization outputs

**Key Visualizations:**
- Performance comparison charts
- Calibration curves
- Confusion matrices
- Subgroup performance analysis
- Multi-dimensional radar chart

This evaluation provides a holistic view of model performance beyond simple accuracy,
enabling responsible and informed model selection decisions.